In [162]:
import pyautogui as pyg
pyg.sleep(2)
basex=408
basey=491
gap=50

In [163]:
for row in range(8):
    for col in range(8):
        x = basex + col * gap
        y = basey + row * gap
        color = pyg.pixel(x, y)
        print(row, col, x, y, color)
    

0 0 408 491 (18, 19, 20)
0 1 458 491 (18, 19, 20)
0 2 508 491 (18, 19, 20)
0 3 558 491 (18, 19, 20)
0 4 608 491 (18, 19, 20)
0 5 658 491 (18, 19, 20)
0 6 708 491 (18, 19, 20)
0 7 758 491 (18, 19, 20)
1 0 408 541 (18, 19, 20)
1 1 458 541 (18, 19, 20)
1 2 508 541 (18, 19, 20)
1 3 558 541 (18, 19, 20)
1 4 608 541 (18, 19, 20)
1 5 658 541 (18, 19, 20)
1 6 708 541 (18, 19, 20)
1 7 758 541 (18, 19, 20)
2 0 408 591 (18, 19, 20)
2 1 458 591 (18, 19, 20)
2 2 508 591 (18, 19, 20)
2 3 558 591 (18, 19, 20)
2 4 608 591 (18, 19, 20)
2 5 658 591 (18, 19, 20)
2 6 708 591 (18, 19, 20)
2 7 758 591 (18, 19, 20)
3 0 408 641 (18, 19, 20)
3 1 458 641 (18, 19, 20)
3 2 508 641 (18, 19, 20)
3 3 558 641 (18, 19, 20)
3 4 608 641 (18, 19, 20)
3 5 658 641 (18, 19, 20)
3 6 708 641 (18, 19, 20)
3 7 758 641 (18, 19, 20)
4 0 408 691 (18, 19, 20)
4 1 458 691 (18, 19, 20)
4 2 508 691 (18, 19, 20)
4 3 558 691 (18, 19, 20)
4 4 608 691 (18, 19, 20)
4 5 658 691 (18, 19, 20)
4 6 708 691 (18, 19, 20)
4 7 758 691 (18, 19, 20)


In [164]:
def get_gem_position(r, c):
    x = basex + c * gap
    y = basey + r * gap
    return (x, y)

In [165]:
import colorsys

def get_gem_color(r, c, im):
    x, y = get_gem_position(r, c)
    pixels = [im.getpixel((px, py))[:3]
              for px in range(x - 2, x + 3)
              for py in range(y - 2, y + 3)]
    return tuple(round(sum(pixel[k] for pixel in pixels) / len(pixels)) for k in range(3))

def same_gem_color(color1, color2, hue_tolerance=0.04):
    hsv1 = colorsys.rgb_to_hsv(*(channel / 255 for channel in color1[:3]))
    hsv2 = colorsys.rgb_to_hsv(*(channel / 255 for channel in color2[:3]))
    if hsv1[1] < 0.16 or hsv2[1] < 0.16:
        return hsv1[1] < 0.16 and hsv2[1] < 0.16 and abs(hsv1[2] - hsv2[2]) <= 0.12
    hue_delta = abs(hsv1[0] - hsv2[0])
    return min(hue_delta, 1 - hue_delta) <= hue_tolerance

In [166]:

def can_move_left(r, c, im):
    if not (0 <= r < 8 and 1 <= c < 8):
        return False

    left_c = c - 1
    left_color = get_gem_color(r, left_c, im)
    current_color = get_gem_color(r, c, im)

    def color_after_swap(row, col):
        if (row, col) == (r, left_c):
            return current_color
        if (row, col) == (r, c):
            return left_color
        return get_gem_color(row, col, im)

    def makes_match(row, col):
        color = color_after_swap(row, col)
        for dr, dc in ((0, 1), (1, 0)):
            count = 1
            for direction in (-1, 1):
                nr = row + dr * direction
                nc = col + dc * direction
                while (0 <= nr < 8 and 0 <= nc < 8
                       and same_gem_color(color_after_swap(nr, nc), color)):
                    count += 1
                    nr += dr * direction
                    nc += dc * direction
            if count >= 3:
                return True
        return False

    return makes_match(r, left_c) or makes_match(r, c)

In [167]:

def _can_swap_and_match(r, c, dr, dc, im):
    nr = r + dr
    nc = c + dc
    if not (0 <= r < 8 and 0 <= c < 8 and 0 <= nr < 8 and 0 <= nc < 8):
        return False

    current_color = get_gem_color(r, c, im)
    neighbor_color = get_gem_color(nr, nc, im)

    def color_after_swap(row, col):
        if (row, col) == (r, c):
            return neighbor_color
        if (row, col) == (nr, nc):
            return current_color
        return get_gem_color(row, col, im)

    def makes_match(row, col):
        color = color_after_swap(row, col)
        for axis_r, axis_c in ((0, 1), (1, 0)):
            count = 1
            for direction in (-1, 1):
                check_r = row + axis_r * direction
                check_c = col + axis_c * direction
                while (0 <= check_r < 8 and 0 <= check_c < 8
                       and same_gem_color(color_after_swap(check_r, check_c), color)):
                    count += 1
                    check_r += axis_r * direction
                    check_c += axis_c * direction
            if count >= 3:
                return True
        return False

    return makes_match(r, c) or makes_match(nr, nc)

def can_move_right(r, c, im):
    return _can_swap_and_match(r, c, 0, 1, im)

def can_move_up(r, c, im):
    return _can_swap_and_match(r, c, -1, 0, im)

def can_move_down(r, c, im):
    return _can_swap_and_match(r, c, 1, 0, im)

In [168]:
im = pyg.screenshot()
print(can_move_left(5, 4, im))

True


In [169]:
print(get_gem_position(0,0))
print(get_gem_position(0,1))
print(get_gem_position(1,0))

(408, 491)
(458, 491)
(408, 541)


In [170]:
im = pyg.screenshot()
for i in range(8):
    x, y = get_gem_position(0, i)
    print(get_gem_color(0, i, im))

(18, 19, 20)
(18, 19, 20)
(18, 19, 20)
(18, 19, 20)
(18, 19, 20)
(18, 19, 20)
(18, 19, 20)
(18, 19, 20)


In [171]:
import time

def rgb_distance(color1, color2):
    return sum((int(color1[k]) - int(color2[k])) ** 2 for k in range(3)) ** 0.5

def changed_gem_count(before, after):
    changed = 0
    for row in range(8):
        for col in range(8):
            if rgb_distance(get_gem_color(row, col, before), get_gem_color(row, col, after)) > 25:
                changed += 1
    return changed

def count_board_colors(im):
    palette = []
    for row in range(8):
        for col in range(8):
            x, y = get_gem_position(row, col)
            if x < 2 or y < 2 or x + 2 >= im.width or y + 2 >= im.height:
                return []
            color = get_gem_color(row, col, im)
            if not any(same_gem_color(color, known) for known in palette):
                palette.append(color)
    return palette

start_time = time.time()
move_count = 0
no_move_reported = False
stop_after_failed_move = False
directions = ((can_move_left, 0, -1), (can_move_right, 0, 1),
              (can_move_up, -1, 0), (can_move_down, 1, 0))
print("60 秒計時開始；請確認遊戲畫面在前景，棋盤座標已設定正確。")
im = pyg.screenshot()

while time.time() - start_time < 60:
    im = pyg.screenshot()
    palette = count_board_colors(im)
    if len(palette) < 3:
        print(f"棋盤辨識失敗：只辨識到 {len(palette)} 種顏色；檢查遊戲視窗與 basex/basey/gap。")
        break

    moved = False
    for r in range(8):
        for c in range(8):
            for can_move, dr, dc in directions:
                if time.time() - start_time >= 60:
                    break
                if not can_move(r, c, im):
                    continue

                x1, y1 = get_gem_position(r, c)
                x2, y2 = get_gem_position(r + dr, c + dc)
                previous_im = im
                print(f"交換：({r}, {c}) -> ({r + dr}, {c + dc})")
                pyg.moveTo(x1, y1)
                try:
                    pyg.dragTo(x2, y2, duration=0.35, button="left")
                    pyg.sleep(0.1)
                finally:
                    pyg.mouseUp()

                pyg.sleep(0.5)
                im = pyg.screenshot()
                changed = changed_gem_count(previous_im, im)
                if changed < 3:
                    print(f"交換後只有 {changed} 格顏色明顯改變；停止以避免無效重複操作。")
                    stop_after_failed_move = True
                else:
                    move_count += 1
                    moved = True
                    print(f"棋盤已更新 {changed} 格，完成第 {move_count} 次交換。")
                break
            if moved or stop_after_failed_move or time.time() - start_time >= 60:
                break
        if moved or stop_after_failed_move or time.time() - start_time >= 60:
            break

    if stop_after_failed_move:
        break
    if moved:
        no_move_reported = False
        continue
    if not no_move_reported:
        print("目前沒有找到有效步，繼續等待棋盤更新。")
        no_move_reported = True
    pyg.sleep(0.2)

print(f"60 秒結束，共執行 {move_count} 次交換。")

60 秒計時開始；請確認遊戲畫面在前景，棋盤座標已設定正確。
棋盤辨識失敗：只辨識到 1 種顏色；檢查遊戲視窗與 basex/basey/gap。
60 秒結束，共執行 0 次交換。
